In [1]:
import os
import sys
import re

# =========================================================
# 1. ÉP VERSION PYSPARK (PHẢI CHẠY ĐẦU TIÊN TRƯỚC KHI IMPORT)
# =========================================================
modules_to_remove = [mod for mod in sys.modules if mod.startswith('pyspark') or mod.startswith('py4j')]
for mod in modules_to_remove: 
    del sys.modules[mod]

sys.path = [p for p in sys.path if "/usr/local/spark" not in p]
if "PYTHONPATH" in os.environ: 
    del os.environ["PYTHONPATH"]
    
conda_site_packages = "/opt/conda/lib/python3.13/site-packages"
if conda_site_packages not in sys.path: 
    sys.path.insert(0, conda_site_packages)
    
os.environ["SPARK_HOME"] = os.path.join(conda_site_packages, "pyspark")
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"


# =========================================================
# 2. BÂY GIỜ MỚI IMPORT PYSPARK VÀ KHỞI TẠO SESSION
# =========================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, udf
from pyspark.sql.types import FloatType

spark = SparkSession.builder \
    .appName("textBlob_token_Pipeline") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.secret.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "hadoop") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://raw-financial-data/iceberg_warehouse") \
    .getOrCreate()


# =========================================================
# 3. VÁ LỖI THỜI GIAN HADOOP
# =========================================================
hadoop_conf = spark._jsc.hadoopConfiguration()
iterator = hadoop_conf.iterator()
while iterator.hasNext():
    entry = iterator.next()
    val = str(entry.getValue()).strip().lower()
    match = re.fullmatch(r"(\d+)([smhd])", val)
    if match:
        num, unit = int(match.group(1)), match.group(2)
        ms_val = num * 1000 if unit == 's' else num * 60000 if unit == 'm' else num * 3600000 if unit == 'h' else num * 86400000
        hadoop_conf.set(entry.getKey(), str(ms_val))

print("✅ Khởi tạo Spark và môi trường hoàn tất!")

✅ Khởi tạo Spark và môi trường hoàn tất!


In [2]:
from textblob import TextBlob

#file parquet nằm trong iceberg-warehouse
df_tokens = spark.read.table("my_catalog.processed_zone_finnhub.news_tokens")
df_tokens.show(5, truncate=False)

+---------+-------------------+---------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|id       |published_at       |title_tokens                                                                                       |summary_tokens                                                                                                                                            

In [3]:
# 1. Định nghĩa các hàm xử lý TextBlob (Giữ nguyên logic kiểm tra Null của bạn)
def get_polarity(text):
    if text is None or text == "":
        return 0.0
    return TextBlob(text).sentiment.polarity

def get_subjectivity(text):
    if text is None or text == "":
        return 0.0
    return TextBlob(text).sentiment.subjectivity

# Đăng ký UDF với PySpark
udf_polarity = udf(get_polarity, FloatType()) ##(Độ phân cực / Sắc thái cảm xúc)
udf_subjectivity = udf(get_subjectivity, FloatType()) ## Subjectivity (Tính chủ quan)

# 2. Nối các phần tử mảng (Array) thành Chuỗi (String) cho cả Title và Summary
# Định dạng concat_ws sẽ tự động bỏ qua các phần tử null nếu có trong mảng
df_processed = df_tokens \
    .withColumn("title_text", concat_ws(" ", col("title_tokens"))) \
    .withColumn("summary_text", concat_ws(" ", col("summary_tokens")))


# 3. Áp dụng UDF để tính điểm độc lập cho tiêu đề và tóm tắt
df_final = df_processed \
    .withColumn("title_polarity", udf_polarity(col("title_text"))) \
    .withColumn("title_subjectivity", udf_subjectivity(col("title_text"))) \
    .withColumn("summary_polarity", udf_polarity(col("summary_text"))) \
    .withColumn("summary_subjectivity", udf_subjectivity(col("summary_text")))

# 4. Kiểm tra cấu trúc phân lớp điểm số vừa tạo
df_final.select(
    "title_text", 
    "title_polarity", 
    "title_subjectivity",
    "summary_text", 
    "summary_polarity", 
    "summary_subjectivity"
).show(5, truncate=50) # Giới hạn độ dài hiển thị chuỗi là 50 ký tự cho đỡ rối mắt


+--------------------------------------------------+--------------+------------------+--------------------------------------------------+----------------+--------------------+
|                                        title_text|title_polarity|title_subjectivity|                                      summary_text|summary_polarity|summary_subjectivity|
+--------------------------------------------------+--------------+------------------+--------------------------------------------------+----------------+--------------------+
|market chatter jpmorgan google firms asked pres...|           0.0|               0.0|jpmorgan chase jpm alphabet goog googl google d...|             0.0|                 0.0|
|uk regulator sets new rules google search boost...|    0.13636364|        0.45454547|britain competition watchdog ordered google pro...|       0.1590909|          0.48863637|
|google cloud model garden platform exclusive cu...|           0.0|               0.0|google cloud summit london model g

In [4]:
from pyspark.sql.functions import col, when

# 1. Định nghĩa các mốc điểm (Threshold) phù hợp với bản chất từng thang điểm
# Đối với Polarity (Cảm xúc)
POLARITY_POS_THRES = 0.1
POLARITY_NEG_THRES = -0.1

# Đối với Subjectivity (Tính chủ quan: từ 0.0 đến 1.0)
# Thường mốc trung vị 0.5 được chọn để phân tách Khách quan và Chủ quan
SUBJECTIVITY_THRES = 0.5

# 2. Tạo cột nhãn cảm xúc cho Title
df_labeled = df_final.withColumn(
    "title_polarity_sentiment",
    when(col("title_polarity") >= POLARITY_POS_THRES, "Tích cực 🟢")
    .when(col("title_polarity") <= POLARITY_NEG_THRES, "Tiêu cực 🔴")
    .otherwise("Trung tính ⚪")
)

# 3. Kế thừa từ df_labeled trước đó để tạo nhãn cảm xúc cho Summary (Tránh ghi đè)
df_labeled = df_labeled.withColumn(
    "summary_polarity_sentiment",
    when(col("summary_polarity") >= POLARITY_POS_THRES, "Tích cực 🟢")
    .when(col("summary_polarity") <= POLARITY_NEG_THRES, "Tiêu cực 🔴")
    .otherwise("Trung tính ⚪")
)

# 4. Phân loại tính chủ quan cho Title (Sử dụng đúng thang điểm dương)
df_labeled = df_labeled.withColumn(
    "title_subjectivity_sentiment",
    when(col("title_subjectivity") >= SUBJECTIVITY_THRES, "Chủ quan 🧠")
    .otherwise("Khách quan 📊")
)

# 5. Phân loại tính chủ quan cho Summary
df_labeled = df_labeled.withColumn(
    "summary_subjectivity_sentiment",
    when(col("summary_subjectivity") >= SUBJECTIVITY_THRES, "Chủ quan 🧠")
    .otherwise("Khách quan 📊")
)

# 6. Hiển thị đối chiếu kết quả (Sửa chính xác tên các cột đã khởi tạo)
df_labeled.select(
    "title_polarity", "title_polarity_sentiment",
    "summary_polarity", "summary_polarity_sentiment",
    "title_subjectivity", "title_subjectivity_sentiment",
    "summary_subjectivity", "summary_subjectivity_sentiment"
).show(50, truncate=False)

+--------------+------------------------+----------------+--------------------------+------------------+----------------------------+--------------------+------------------------------+
|title_polarity|title_polarity_sentiment|summary_polarity|summary_polarity_sentiment|title_subjectivity|title_subjectivity_sentiment|summary_subjectivity|summary_subjectivity_sentiment|
+--------------+------------------------+----------------+--------------------------+------------------+----------------------------+--------------------+------------------------------+
|0.0           |Trung tính ⚪            |0.0             |Trung tính ⚪              |0.0               |Khách quan 📊               |0.0                 |Khách quan 📊                 |
|0.13636364    |Tích cực 🟢             |0.1590909       |Tích cực 🟢               |0.45454547        |Khách quan 📊               |0.48863637          |Khách quan 📊                 |
|0.0           |Trung tính ⚪            |0.04            |Trung tính ⚪      

In [5]:
# 1. Bắt buộc phải có TÊN CATALOG ở đầu (ví dụ: my_catalog)
# Hãy thay 'my_catalog' bằng đúng tên catalog bạn đã cấu hình trong Spark
iceberg_table_path = "my_catalog.processed_zone_finnhub.news_textBlob_tokens_sentiment_score"

print(f"Đang ghi dữ liệu vào bảng Iceberg: {iceberg_table_path}...")

# 2. Sử dụng API tiêu chuẩn với format("iceberg") và saveAsTable
df_labeled.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable(iceberg_table_path)

print("Lưu dữ liệu vào Iceberg thành công!")


Đang ghi dữ liệu vào bảng Iceberg: my_catalog.processed_zone_finnhub.news_textBlob_tokens_sentiment_score...
Lưu dữ liệu vào Iceberg thành công!
